In [6]:
import pandas as pd
import nltk
import re
nltk.download('stopwords')
from nltk.corpus import stopwords

data = {
    'ticket_text': [
        "My payment failed and I was charged twice",
        "App crashes when I open settings",
        "How do I reset my password?",
        "Cannot login, account locked",
        "Slow loading on dashboard page",
        "Refund not received after 7 days",
        "App not working on iPhone",
        "Need help changing my email address"
    ],
    'category': ['billing','bug','account','account','performance','billing','bug','account'],
    'priority': ['high','high','low','medium','medium','high','medium','low']
}
df = pd.DataFrame(data)
print(df)

                                 ticket_text     category priority
0  My payment failed and I was charged twice      billing     high
1           App crashes when I open settings          bug     high
2                How do I reset my password?      account      low
3               Cannot login, account locked      account   medium
4             Slow loading on dashboard page  performance   medium
5           Refund not received after 7 days      billing     high
6                  App not working on iPhone          bug   medium
7        Need help changing my email address      account      low


[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\jatot\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


In [7]:
def clean_text(text):
    text = text.lower()
    text = re.sub(r'[^a-z\s]', '', text)
    tokens = text.split()
    tokens = [w for w in tokens if w not in stopwords.words('english')]
    return ' '.join(tokens)

df['cleaned'] = df['ticket_text'].apply(clean_text)
print(df['cleaned'])


0        payment failed charged twice
1           app crashes open settings
2                      reset password
3         cannot login account locked
4         slow loading dashboard page
5                refund received days
6                  app working iphone
7    need help changing email address
Name: cleaned, dtype: str


In [8]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.pipeline import Pipeline

category_model = Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])
priority_model = Pipeline([('tfidf', TfidfVectorizer()), ('clf', MultinomialNB())])

category_model.fit(df['cleaned'], df['category'])
priority_model.fit(df['cleaned'], df['priority'])
print("Models trained successfully!")


Models trained successfully!


In [9]:
def classify_ticket(text):
    cleaned = clean_text(text)
    return {
        'category': category_model.predict([cleaned])[0],
        'priority': priority_model.predict([cleaned])[0]
    }

print(classify_ticket("I was charged twice for my subscription"))


{'category': np.str_('billing'), 'priority': np.str_('high')}
